# Run all batches

Delivers every batch found in `landing/` (any number, no fixed list) **one at a time**, the way a sender would: write the `_SUCCESS` marker last.
EventBridge starts the workflow by itself. This notebook only writes the marker and waits for the result.
Then it checks the gold tables and the KPI views.

One at a time, because every run merges into the same Iceberg tables (see `docs/phase_07_step_functions`).

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Works whether Jupyter starts in the project root or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "infra"))

import sql_files
from aws_helpers import (REGION, SETTINGS, account_id, aws_json, batches_in_landing, files_in_batch, ingest_date_of, release_batch,
                         run_athena, whoami)

BATCHES = batches_in_landing()          # every batch folder that exists, in ascending order
MACHINE_ARN = f"arn:aws:states:{REGION}:{account_id()}:stateMachine:{SETTINGS['state_machine_name']}"
GOLD = SETTINGS["glue_databases"]["gold"]

print("identity:", whoami())
for batch in BATCHES:
    print(f"batch {batch}: {len(files_in_batch(batch))} data files in landing/batch_{batch:02d}/")

## 2. Release the batches
For each batch: write the marker, wait for the run, print how it ended. Each run takes about 9 minutes (most of it Athena
checking once a minute whether a merge has finished). A run that does not end `SUCCEEDED` stops the notebook.
Releasing a batch that was loaded before is fine: every step overwrites its own result.

In [ ]:
import time

runs = {}
for batch in BATCHES:
    started = time.time()
    run = release_batch(batch, MACHINE_ARN)
    runs[batch] = run
    print(f"batch {batch}: {run['status']:<10} {(time.time() - started) / 60:4.1f} min   output {run.get('output')}")
    assert run["status"] == "SUCCEEDED", (run.get("error"), run.get("cause"))
print()
print("OK  all", len(runs), "batches ran through the event trigger")

## 3. The gold tables

In [ ]:
def count(sql):
    return int(run_athena(sql)["rows"][0][0])


GOLD_COUNTS = {table: count(f"SELECT COUNT(*) FROM {GOLD}.{table}")
               for table in ("dim_patient", "dim_facility", "dim_practitioner", "dim_date", "fact_encounter")}
for table, rows in GOLD_COUNTS.items():
    print(f"{table:<18}{rows:>8}")

classes = run_athena(f"SELECT encounter_class, COUNT(*) FROM {GOLD}.fact_encounter GROUP BY 1 ORDER BY 2 DESC")["rows"]
print()
print("encounters by class:", {name: int(rows) for name, rows in classes})

# Gold must hold exactly the distinct keys of silver, whatever the number of batches
SILVER = SETTINGS["glue_databases"]["silver"]
SILVER_KEYS = {
    "dim_patient": f"SELECT COUNT(DISTINCT patient_id) FROM {SILVER}.patient",
    "dim_facility": f"SELECT COUNT(DISTINCT organization_id) FROM {SILVER}.organization",
    "dim_practitioner": f"SELECT COUNT(DISTINCT npi) FROM {SILVER}.practitioner",
    "fact_encounter": f"SELECT COUNT(DISTINCT encounter_id) FROM {SILVER}.encounter",
}
for table, sql in SILVER_KEYS.items():
    assert GOLD_COUNTS[table] == count(sql), table
print("OK  every gold table holds exactly the distinct keys of silver")

## 4. The data-quality checks, for every batch

In [ ]:
CHECKS = sql_files.read_checks(ROOT / "sql" / "dq")
DETAILED = sql_files.detailed_checks_query(list(CHECKS.values()))
for batch in BATCHES:
    ingest_date = ingest_date_of(batch)
    result = run_athena(DETAILED, parameters=[sql_files.quote_date(ingest_date)], timeout=300)
    failing = {name: int(rows) for name, rows in result["rows"] if int(rows)}
    print(f"batch {batch} ({ingest_date}): {len(result['rows'])} checks, failing: {failing or 'none'}")
    assert not failing
print("OK  every check passes for every batch")

## 5. The five KPI views

In [ ]:
VIEWS = {
    "vw_daily_admissions_discharges": "SUM(admissions), SUM(discharges)",
    "vw_avg_los_by_facility_month": "SUM(inpatient_stays), ROUND(SUM(avg_los_days * inpatient_stays) / SUM(inpatient_stays), 2)",
    "vw_readmission_30d": "SUM(inpatient_admissions), SUM(readmissions_30d)",
    "vw_ed_volume_by_hour_dow": "SUM(emergency_visits), COUNT(*)",
    "vw_top_reasons": "COUNT(*), MAX(encounters)",
}
VALUES = {}
for view, expression in VIEWS.items():
    VALUES[view] = run_athena(f"SELECT {expression} FROM {GOLD}.{view}")["rows"][0]
    print(f"{view:<34}{VALUES[view]}")

# The views must add up to the fact table, whatever the number of batches
inpatient = count(f"SELECT COUNT(*) FROM {GOLD}.fact_encounter WHERE is_inpatient")
emergency = count(f"SELECT COUNT(*) FROM {GOLD}.fact_encounter WHERE is_emergency")
admissions, readmissions = (int(value) for value in VALUES["vw_readmission_30d"])
assert admissions == inpatient and int(VALUES["vw_ed_volume_by_hour_dow"][0]) == emergency
print(f"OK  {inpatient} inpatient stays, {readmissions} of them readmissions within 30 days; {emergency} emergency visits")

## 6. The runs, as Step Functions shows them

In [ ]:
for e in aws_json("stepfunctions", "list-executions", "--state-machine-arn", MACHINE_ARN, "--max-results", "6")["executions"]:
    print(f"{e['startDate'][:19]}  {e['status']:<10}  {e['name'][:40]}")
print()
print(f"Step Functions console > {SETTINGS['state_machine_name']} > Executions. Charts of the KPIs: notebooks/kpi_analysis.ipynb")

## What this is for
| Item | Where |
|---|---|
| Deliver a batch | write `landing/batch_NN/_SUCCESS` (`release_batch` in `infra/aws_helpers.py` does it and waits) |
| Run one batch again | write its marker again; the result is the same |
| Look at the results | `kpi_analysis.ipynb` (charts), Athena console (`pf_gold` views) |
| Next | Phase 11 moves the AWS resources into CloudFormation; the same notebook then re-runs the batches on the new resources |